# DADS5001 Mini Project
## Exploratory Data Analysis of Government Construction Procurement

Notebook นี้วิเคราะห์ข้อมูลโครงการจ้างก่อสร้างภาครัฐ
ปีงบประมาณ 2569 เพื่อทำความเข้าใจลักษณะของโครงการ
การกระจายของมูลค่า วิธีจัดซื้อ หน่วยงาน พื้นที่ และผู้รับจ้าง

### Main Questions

1. โครงการจ้างก่อสร้างมีมูลค่ากระจายตัวอย่างไร
2. ราคาที่ตกลงแตกต่างจากวงเงินงบประมาณและราคากลางเพียงใด
3. รูปแบบของราคาแตกต่างกันตามวิธีจัดซื้อจัดจ้างหรือไม่
4. หน่วยงานและพื้นที่ใดมีจำนวนหรือมูลค่าโครงการสูง
5. ผู้รับจ้างมีการกระจุกตัวมากน้อยเพียงใด
6. มีรูปแบบใดที่ควรนำไปตรวจสอบเพิ่มเติม

### Scope

- ใช้ข้อมูลเฉพาะงาน `จ้างก่อสร้าง` ปีงบประมาณ 2569
- วิเคราะห์ภาพรวมที่ระดับโครงการโดยใช้ `รหัสโครงการ`
- ใช้ข้อมูลทุกแถวเมื่อต้องวิเคราะห์ผู้ชนะหรือรายละเอียดสัญญา
- เน้น Pandas, NumPy, Matplotlib และ Seaborn
- ยังไม่กำหนด Red Flag จนกว่าจะเห็นผลจาก EDA

> รูปแบบที่ผิดสังเกตเป็นเพียงสัญญาณที่ควรตรวจสอบเพิ่มเติม
> ไม่ใช่หลักฐานยืนยันการทุจริต

## วิธีรันเวอร์ชัน GitHub

เปิด Notebook นี้ใน Google Colab แล้วเลือก **Runtime → Run all** ข้อมูลจะอ่านจาก [data_for_github](https://drive.google.com/drive/folders/1ssVrUcY4TiYee9T2B0pwgr5SwvPp_lAq) โดยดาวน์โหลดเฉพาะไฟล์ที่ใช้ลงพื้นที่ชั่วคราวของ Runtime ไม่ต้องตั้งโฟลเดอร์ผลลัพธ์หรือเชื่อม Drive ส่วนตัว

- กราฟและตารางแสดงใน Notebook ไม่มีการส่งออก CSV/PNG/SVG ไปยัง Drive หรือเครื่องของผู้อ่าน
- ไฟล์ข้อมูล ฟอนต์ แผนที่ และ CSV พักที่จำเป็นอยู่ใน Runtime เท่านั้น เซลล์สุดท้ายลบไฟล์พักและเก็บ DataFrame ไว้ให้วิเคราะห์ต่อ หากหยุดกลางทางให้รัน `cleanup_runtime()` เมื่อไม่ต้องใช้ไฟล์พักแล้ว
- รันแต่ละ Notebook ใน Runtime แยกกันเพื่อลด RAM; หาก RAM ใช้ถึง 80% โค้ดจะหยุดพร้อมคำแนะนำ แทนการฝืนประมวลผล
- กรณี Drive จำกัดการดาวน์โหลด ให้ดาวน์โหลดข้อมูลเองและระบุ `LOCAL_DATA_DIR` ในเซลล์ตั้งค่า
- สูตร เกณฑ์ ตัวกรอง และลำดับข้อมูลคงตามไฟล์ต้นฉบับ อ่านข้อแตกต่างระหว่าง Notebook และรายการไฟล์ใน [DATA_FILES.md](../DATA_FILES.md)

ผลลัพธ์เดิมที่ฝังในไฟล์ถูกล้างเพื่อให้ผู้อ่านเห็นผลจากการรันครั้งใหม่


In [ ]:
# GitHub / Colab: ดาวน์โหลดข้อมูลเฉพาะที่ใช้ลงพื้นที่ชั่วคราว ไม่ต้อง Mount Drive
# เปลี่ยนเป็น Path โฟลเดอร์ข้อมูลที่มีอยู่แล้วได้ เพื่อไม่ต้องดาวน์โหลดซ้ำ
LOCAL_DATA_DIR = None
import atexit
import gc
import hashlib
import importlib.util
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# ติดตั้งเฉพาะแพ็กเกจที่ยังไม่มี; subprocess นี้รอจนจบ ไม่มี worker ค้าง
_packages = {"pandas": "pandas>=2.2,<3", "numpy": "numpy", "matplotlib": "matplotlib", "seaborn": "seaborn", "geopandas": "geopandas", "psutil": "psutil", "gdown": "gdown", "IPython": "ipython"}
_missing_packages = [requirement for module, requirement in _packages.items() if importlib.util.find_spec(module) is None]
if _missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing_packages], check=True)
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gdown
from IPython.display import display

def check_memory(stage="ประมวลผล"):
    # หยุดอย่างชัดเจนเมื่อ RAM สูง แทนการฝืนจน Runtime หลุด
    if psutil.virtual_memory().percent >= 80:
        gc.collect()
        if psutil.virtual_memory().percent >= 80:
            raise MemoryError(f"RAM ใช้เกิน 80% ก่อน {stage}; กรุณา Restart runtime หรือใช้ High-RAM แล้ว Run all ใหม่")

# รัน setup ซ้ำจะปิดพื้นที่ชั่วคราวของ Notebook รอบก่อน
if "_github_runtime" in globals():
    _github_runtime.cleanup()
_github_runtime = tempfile.TemporaryDirectory(prefix="procurement_github_")
atexit.register(_github_runtime.cleanup)
RUNTIME_DIR = Path(_github_runtime.name)
DATA_DIR = RUNTIME_DIR / "inputs"
DATA_DIR.mkdir()
_initial_figure_numbers = set(plt.get_fignums())

DATA_FILES = {
    "07_lower10_project_budget_2567.csv": {
        "id": "1YQpAhUcghJ9gM6eIC6qBZGe6WcVuKyWR",
        "size": 4169
    },
    "07_lower10_project_budget_2568.csv": {
        "id": "1aNYIMOH7Da8VvVhXI1XQigBYRNWk9IHj",
        "size": 5098
    },
    "07_lower10_project_budget_2569.csv": {
        "id": "1tU1V9n2ysJmXxvmnm_SPz7n7Bhbz2B5_",
        "size": 4371
    },
    "07_top10_project_budget_2567.csv": {
        "id": "1bkMYSpLM4UKivKCHe7S8Dsho0BaLgCdJ",
        "size": 5619
    },
    "07_top10_project_budget_2568.csv": {
        "id": "1xsItnomfe9q4A9NYJ5U_lR0764mZMEE4",
        "size": 5682
    },
    "07_top10_project_budget_2569.csv": {
        "id": "1ZwOBFVQkTRAnazjoeXzr3jkpSGFl2bkI",
        "size": 6385
    },
    "2569-egp-contract-1.csv": {
        "id": "1ygmuKsvVjNpjfg0k94Olcuox8efDP3cY",
        "size": 615601896
    },
    "2569-egp-contract-2.csv": {
        "id": "1DUAh8zAtUUV4Hm2yRplCryUOQDFb3Ad8",
        "size": 542097077
    },
    "2569-egp-contract-3.csv": {
        "id": "1fD_6GqtIY7LUempYqId0qIEldcJd2moM",
        "size": 529193616
    },
    "2569-egp-contract-4.csv": {
        "id": "19C4Ak8XNMvm2IBsr7TVpfexZE_1WBke5",
        "size": 526729581
    },
    "2569-egp-contract-5.csv": {
        "id": "1nsV7vqF8Um-QWKX_Q-kEQq6Z1BFDbK_y",
        "size": 521850128
    },
    "2569-egp-contract-6.csv": {
        "id": "1-0zhDZjF2chjeXuPoSVGQWNYtp8fVIzt",
        "size": 522697146
    },
    "2569-egp-contract-7.csv": {
        "id": "1gl9sx_1ve-n7e9M_mQaj0A7mMnk2P0yb",
        "size": 517466336
    },
    "2569-egp-contract-8.csv": {
        "id": "1z0icf65DY2e7WPmc_qV5PPLdUVyT1wdR",
        "size": 455588827
    },
    "construction_contract_review_indicators_2569.csv": {
        "id": "1nYgjub4kfLTyX_9gnz3dM2GZWUM67wRJ",
        "size": 247613515
    },
    "construction_contract_supplier_study_scope_2569.csv": {
        "id": "1dXH9N4ukZAabRQwl9nkQ-KidB1e80ZAh",
        "size": 243311179
    },
    "priority_review_contracts_2569.csv": {
        "id": "1prDQwSgkNCFqarTx0rkFuQK12Q5J2MT_",
        "size": 108598
    },
    "project_overview_2567.csv": {
        "id": "1hEr6tXWRBgEM-uK0dOEKvzIhBdKzHtnx",
        "size": 1685455822
    },
    "project_overview_2568.csv": {
        "id": "1WVh1etQtH45AemJE4J7mBCDOOT9QmQff",
        "size": 1448461110
    },
    "project_overview_2569.csv": {
        "id": "1h3Oah1ARYenycAJnLa1DTefzjMryQuJ5",
        "size": 1227718817
    },
    "repeated_near_500k_agency_supplier_2569.csv": {
        "id": "10fT6PqZbBa5mS4GEHHEOf8_qIeASToQk",
        "size": 364037
    }
}

def data_file(name):
    # ใช้ไฟล์ในเครื่องก่อนถ้าผู้ใช้ระบุโฟลเดอร์ โดยไม่แก้ไขไฟล์ต้นฉบับ
    if LOCAL_DATA_DIR is not None:
        candidate = Path(LOCAL_DATA_DIR) / name
        if candidate.is_file():
            return candidate
    if name not in DATA_FILES:
        raise FileNotFoundError(f"ยังไม่มีข้อมูล {name} ในชุดที่แชร์ กรุณาตรวจ DATA_FILES หรือ LOCAL_DATA_DIR")
    item = DATA_FILES[name]
    target = DATA_DIR / name
    if target.is_file():
        return target
    required_bytes = int(item["size"])
    if shutil.disk_usage(DATA_DIR).free < required_bytes + 512 * 1024**2:
        raise OSError(f"พื้นที่ชั่วคราวไม่พอสำหรับ {name}")
    partial = target.with_suffix(target.suffix + ".part")
    print(f"อ่านข้อมูล {name} ({required_bytes / 1024**2:,.1f} MB)")
    try:
        result = gdown.download(id=item["id"], output=str(partial), quiet=False)
        if result is None or not partial.is_file() or partial.stat().st_size != required_bytes:
            raise IOError(f"ดาวน์โหลด {name} ไม่ครบ หรือไฟล์ใน Drive เปลี่ยนรุ่น")
        if item.get("md5"):
            digest = hashlib.md5()
            with partial.open("rb") as stream:
                for block in iter(lambda: stream.read(4 * 1024**2), b""):
                    digest.update(block)
            if digest.hexdigest() != item["md5"]:
                raise IOError(f"Checksum ไม่ตรงสำหรับ {name}")
        partial.replace(target)
    except Exception as exc:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"อ่าน {name} ไม่สำเร็จ: ตรวจสิทธิ์ Anyone with the link ของโฟลเดอร์ข้อมูล หรือดาวน์โหลดเองแล้วกำหนด LOCAL_DATA_DIR") from exc
    return target

def release_download(path):
    # ลบเฉพาะสำเนาที่ Notebook นี้ดาวน์โหลด ไม่ลบไฟล์ใน LOCAL_DATA_DIR
    path = Path(path)
    if path.parent == DATA_DIR:
        path.unlink(missing_ok=True)

def cleanup_runtime():
    # เก็บ DataFrame ผลวิเคราะห์ไว้ให้ย้อนกลับมาดูต่อได้
    for number in set(plt.get_fignums()) - _initial_figure_numbers:
        plt.close(number)
    _github_runtime.cleanup()
    gc.collect()
    print("ปิดรูปและลบไฟล์พักของ Notebook แล้ว; DataFrame ผลวิเคราะห์ยังอยู่ใน RAM")

check_memory("เริ่ม Notebook")

# สร้างข้อมูลก่อสร้างแบบเดียวกับ Notebook 02 เมื่อไฟล์ prepared รุ่นเก่าไม่อยู่ใน Drive
def prepare_construction_file():
    destination = RUNTIME_DIR / "construction_contracts_2569.csv"
    if destination.exists():
        return destination
    first_write = True
    try:
        for name in sorted(n for n in DATA_FILES if n.startswith("2569-egp-contract-") and n.endswith(".csv")):
            path = data_file(name)
            check_memory("อ่านไฟล์ต้นทางเป็น chunk")
            with pd.read_csv(path, chunksize=100_000, low_memory=False) as reader:
                for chunk in reader:
                    check_memory("คัดงานก่อสร้าง")
                    chunk.columns = chunk.columns.str.strip()
                    project_type = chunk["ชื่อประเภทโครงการ"].astype("string").str.strip().fillna("ไม่ระบุ")
                    selected = chunk.loc[project_type.eq("จ้างก่อสร้าง")].copy()
                    if not selected.empty:
                        selected["source_file"] = path.name
                        # CSV พักอยู่ใน TemporaryDirectory เพื่อคงลำดับแถวและ dtype หลังอ่านเหมือนต้นฉบับ
                        selected.to_csv(destination, mode="w" if first_write else "a", header=first_write, index=False, encoding="utf-8-sig")
                        first_write = False
                    del chunk, selected, project_type
            release_download(path)
            gc.collect()
        if first_write:
            raise ValueError("ไม่พบแถวจ้างก่อสร้างในไฟล์ต้นทาง")
    except BaseException:
        destination.unlink(missing_ok=True)
        raise
    return destination


In [ ]:
# GitHub: ใช้ข้อมูลจากลิงก์ที่แชร์ ไม่ต้อง Mount Drive


In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option(
    'display.float_format',
    lambda value: f'{value:,.2f}'
)

sns.set_theme(style='whitegrid')

data_path = prepare_construction_file()

construction_data = pd.read_csv(data_path,low_memory=False)

print(f'Shape: {construction_data.shape}')

## 1. Data Understanding

เริ่มจากตรวจสอบโครงสร้าง ชนิดข้อมูล ค่าว่าง และจำนวนค่าที่ไม่ซ้ำ
ของแต่ละคอลัมน์ เพื่อเลือกตัวแปรที่สามารถนำมาใช้วิเคราะห์ได้จริง

การตรวจในส่วนนี้มุ่งเฉพาะข้อมูลที่จำเป็นต่อ EDA
ไม่ใช่การทำ Data-quality audit อย่างละเอียด

In [ ]:
display(construction_data.head())

In [ ]:
field_summary = pd.DataFrame({
    'column': construction_data.columns,
    'dtype': (
        construction_data
        .dtypes
        .astype(str)
        .values
    ),
    'non_null': (
        construction_data
        .notna()
        .sum()
        .values
    ),
    'missing': (
        construction_data
        .isna()
        .sum()
        .values
    ),
    'missing_pct': (
        construction_data
        .isna()
        .mean()
        .mul(100)
        .values
    ),
    'unique': (
        construction_data
        .nunique(
            dropna=True
        )
        .values
    )
})

display(field_summary)

### Data Understanding Summary

ข้อมูลจ้างก่อสร้างมี 180,079 แถว และ 29 คอลัมน์
ตัวแปรหลักด้านงบประมาณ ราคา วิธีจัดซื้อ หน่วยงาน และพื้นที่
มีข้อมูลค่อนข้างครบถ้วน

คอลัมน์ `ราคากลาง (บาท)` ขาดข้อมูล 107 แถว หรือ 0.06%
และ `วงเงินงบประมาณในสัญญา (บาท)` ขาดข้อมูล 90 แถว
หรือ 0.05% ส่วนข้อมูลพิกัดขาด 1,438 แถว หรือ 0.80%

คอลัมน์ต่อไปนี้มีเพียงค่าเดียว จึงไม่ช่วยในการเปรียบเทียบ
ภายในชุดข้อมูลจ้างก่อสร้าง:

- ชื่อประเภทโครงการ
- ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง
- ปีงบประมาณ
- สถานะโครงการ
- สถานะสัญญา

ก่อนสร้างข้อมูลระดับโครงการ จะตรวจสอบว่าแถวที่ใช้
`รหัสโครงการ` เดียวกันมีค่าของตัวแปรระดับโครงการ
สอดคล้องกันหรือไม่

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
project_columns = [
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อหน่วยงาน',
    'ชื่อหน่วยงานย่อย',
    'ชื่อวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    ),
    'วันที่เกิดรายการ',
    'จังหวัด',
    'เขต/อำเภอ',
    'แขวง/ตำบล'
]

project_consistency = (
    construction_data
    .groupby('รหัสโครงการ')[
        project_columns
    ]
    .nunique(
        dropna=False
    )
)

inconsistent_projects = (
    project_consistency
    .gt(1)
    .sum()
    .rename(
        'projects_with_multiple_values'
    )
    .to_frame()
)

display(inconsistent_projects)

## 2. Project-level Data Preparation

จากการตรวจสอบโครงการที่ปรากฏมากกว่าหนึ่งแถว พบว่าตัวแปร
ระดับโครงการทั้งหมดมีค่าตรงกันภายใน `รหัสโครงการ` เดียวกัน
ได้แก่ ชื่อโครงการ หน่วยงาน วิธีจัดซื้อ วันที่ ราคา และพื้นที่

ดังนั้น สามารถสร้างชุดข้อมูลระดับโครงการโดยเก็บหนึ่งแถวต่อ
`รหัสโครงการ` ได้ โดยไม่ทำให้ข้อมูลระดับโครงการสูญหาย

ชุดข้อมูลที่ใช้ต่อจากนี้แบ่งเป็น:

- `project_data` สำหรับวิเคราะห์จำนวนโครงการ ราคา หน่วยงาน
  วิธีจัดซื้อ และพื้นที่
- `construction_data` สำหรับวิเคราะห์ผู้ชนะและรายละเอียดสัญญา

In [ ]:
project_data = (
    construction_data
    .drop_duplicates(
        subset='รหัสโครงการ',
        keep='first'
    )
    .copy()
)

print(
    f'Contract-level rows: '
    f'{len(construction_data):,}'
)

print(
    f'Project-level rows: '
    f'{len(project_data):,}'
)

print(
    f'Rows removed from project-level analysis: '
    f'{len(construction_data) - len(project_data):,}'
)

In [ ]:
money_columns = [
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    )
]

money_summary = (
    project_data[
        money_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

display(money_summary)

non_positive_summary = pd.DataFrame({
    'column': money_columns,
    'zero_or_negative': [
        project_data[column]
        .le(0)
        .sum()
        for column in money_columns
    ],
    'missing': [
        project_data[column]
        .isna()
        .sum()
        for column in money_columns
    ]
})

display(non_positive_summary)

### Monetary Data Summary

ชุดข้อมูลระดับโครงการมีจำนวน 178,978 โครงการ
และไม่มีวงเงินงบประมาณหรือราคาที่ตกลงเป็นศูนย์หรือติดลบ

วงเงินงบประมาณมีค่ามัธยฐาน 395,000 บาท ขณะที่ค่าเฉลี่ย
อยู่ที่ประมาณ 2.66 ล้านบาท และมีค่าสูงสุดมากกว่า 4 พันล้านบาท
แสดงว่าการกระจายของมูลค่าโครงการมีลักษณะเบ้ขวาอย่างมาก
จากโครงการมูลค่าสูงจำนวนไม่มาก

นอกจากนี้ 75% ของโครงการมีวงเงินงบประมาณไม่เกิน 500,000 บาท
จึงควรสำรวจว่ามีการกระจุกตัวบริเวณระดับมูลค่าบางช่วงหรือไม่

คอลัมน์ราคากลางขาดข้อมูล 105 โครงการ คิดเป็นสัดส่วนต่ำมาก
จึงไม่ตัดโครงการเหล่านี้ออกจากชุดข้อมูลหลัก แต่จะไม่นำมาใช้
เฉพาะในการวิเคราะห์ที่ต้องเปรียบเทียบกับราคากลาง

In [ ]:
budget_column = 'วงเงินงบประมาณ (บาท)'
reference_price_column = 'ราคากลาง (บาท)'

awarded_price_column = (
    'ราคาที่ตกลงซื้อ / จ้าง '
    'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
)

# Positive value means the awarded price is below the budget
project_data['budget_saving'] = (
    project_data[budget_column]
    - project_data[awarded_price_column]
)

project_data['budget_saving_pct'] = (
    project_data['budget_saving']
    .div(project_data[budget_column])
    .mul(100)
)

# Positive value means the awarded price is below the reference price
project_data['reference_discount'] = (
    project_data[reference_price_column]
    - project_data[awarded_price_column]
)

project_data['reference_discount_pct'] = (
    project_data['reference_discount']
    .div(
        project_data[
            reference_price_column
        ]
    )
    .mul(100)
)

In [ ]:
price_difference_columns = [
    'budget_saving',
    'budget_saving_pct',
    'reference_discount',
    'reference_discount_pct'
]

display(
    project_data[
        price_difference_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

price_relationship_summary = pd.DataFrame({
    'comparison': [
        'Awarded price below budget',
        'Awarded price equal to budget',
        'Awarded price above budget',
        'Awarded price below reference price',
        'Awarded price equal to reference price',
        'Awarded price above reference price'
    ],
    'project_count': [
        project_data['budget_saving'].gt(0).sum(),
        project_data['budget_saving'].eq(0).sum(),
        project_data['budget_saving'].lt(0).sum(),
        project_data['reference_discount'].gt(0).sum(),
        project_data['reference_discount'].eq(0).sum(),
        project_data['reference_discount'].lt(0).sum()
    ]
})

price_relationship_summary['project_pct'] = (
    price_relationship_summary['project_count']
    .div(len(project_data))
    .mul(100)
)

display(price_relationship_summary)

### Price Difference Findings

โครงการจำนวน 67.42% มีราคาตกลงต่ำกว่าวงเงินงบประมาณ
ขณะที่ 32.41% มีราคาตกลงเท่ากับวงเงินงบประมาณ และ 0.18%
มีราคาตกลงสูงกว่าวงเงินงบประมาณ

เมื่อเปรียบเทียบกับราคากลาง พบว่า 77.81% ของโครงการ
มีราคาตกลงต่ำกว่าราคากลาง 21.69% มีราคาเท่ากัน และ 0.44%
มีราคาตกลงสูงกว่าราคากลาง

อย่างไรก็ตาม การกระจายของส่วนต่างราคาร้อยละมีค่าผิดสังเกตรุนแรง
โดยเฉพาะ `reference_discount_pct` ซึ่งมีค่าต่ำสุดถึง
-10,399,900% ค่าดังกล่าวอาจเกิดจากราคากลางที่ต่ำผิดปกติ
จนทำให้การหารด้วยราคากลางสร้างค่าร้อยละที่สูงมาก

ดังนั้น การวิเคราะห์จะให้ความสำคัญกับค่ามัธยฐานและ percentile
มากกว่าค่าเฉลี่ย และจะแยกตรวจสอบกรณีราคาตกลงสูงกว่ากรอบราคา
ใน Notebook การวิเคราะห์ Red Flag

## 3. Exploratory Data Analysis

### 3.1 Distribution of Project Budget

การเปรียบเทียบค่าเฉลี่ยและค่ามัธยฐานแสดงให้เห็นว่าวงเงินงบประมาณ
มีการกระจายแบบเบ้ขวา จึงแสดงข้อมูลทั้งในช่วงไม่เกิน percentile
ที่ 99 และในรูป log scale เพื่อให้เห็นโครงสร้างของข้อมูลได้ชัดเจนขึ้น

การจำกัดแกนในกราฟแรกมีไว้เพื่อการมองเห็นเท่านั้น
โดยไม่ได้ลบโครงการมูลค่าสูงออกจากข้อมูล

In [ ]:
project_data['budget_million'] = (
    project_data[budget_column]
    .div(1_000_000)
)

project_data['log10_budget'] = np.log10(
    project_data[budget_column]
)

budget_p99 = (
    project_data['budget_million']
    .quantile(0.99)
)

print(
    f'99th percentile of budget: '
    f'{budget_p99:,.2f} million THB'
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.histplot(
    data=project_data,
    x='budget_million',
    bins=50,
    ax=axes[0],
    color='#4C78A8'
)

axes[0].set_xlim(
    0,
    budget_p99
)

axes[0].set_title(
    'Project Budget Distribution '
    '(Up to 99th Percentile)'
)

axes[0].set_xlabel(
    'Budget (million THB)'
)

axes[0].set_ylabel(
    'Number of projects'
)

sns.histplot(
    data=project_data,
    x='log10_budget',
    bins=50,
    ax=axes[1],
    color='#E67E22'
)

axes[1].set_title(
    'Project Budget Distribution '
    '(Log Scale)'
)

axes[1].set_xlabel(
    'Log10 of budget (THB)'
)

axes[1].set_ylabel(
    'Number of projects'
)

plt.tight_layout()
plt.show()

In [ ]:
common_budget_values = (
    project_data[budget_column]
    .value_counts()
    .head(15)
    .rename_axis('budget')
    .reset_index(name='project_count')
)

common_budget_values['budget_million'] = (
    common_budget_values['budget']
    .div(1_000_000)
)

common_budget_values['project_pct'] = (
    common_budget_values['project_count']
    .div(len(project_data))
    .mul(100)
)

display(
    common_budget_values[
        [
            'budget',
            'budget_million',
            'project_count',
            'project_pct'
        ]
    ]
)

Budget Distribution Findings

วงเงินงบประมาณของโครงการจ้างก่อสร้างมีการกระจายแบบเบ้ขวา
อย่างชัดเจน โดย 76.34% ของโครงการมีวงเงินไม่เกิน 500,000 บาท
ขณะที่ 99% มีวงเงินไม่เกิน 32.50 ล้านบาท แต่ยังมีโครงการ
มูลค่าสูงมากบางรายการซึ่งทำให้ค่าเฉลี่ยสูงกว่าค่ามัธยฐานมาก

นอกจากนี้ พบว่าวงเงินที่ปรากฏบ่อยที่สุดคือ 500,000 บาท
จำนวน 6,523 โครงการ และพบโครงการจำนวนมากที่มีวงเงิน
ใกล้เคียงแต่ต่ำกว่า 500,000 บาท เช่น 499,000, 498,000,
497,000, 496,000, 495,000 และ 490,000 บาท

รูปแบบดังกล่าวแสดงถึงการกระจุกตัวของวงเงินบริเวณ 500,000 บาท
แต่ยังไม่สามารถอธิบายสาเหตุได้จากข้อมูลชุดนี้เพียงอย่างเดียว
จึงควรนำไปเปรียบเทียบกับวิธีจัดซื้อ หน่วยงาน และลักษณะโครงการ
ในขั้นต่อไป

In [ ]:
budget_bins = [
    0,
    100_000,
    200_000,
    300_000,
    400_000,
    500_000,
    1_000_000,
    5_000_000,
    10_000_000,
    50_000_000,
    np.inf
]

budget_labels = [
    '≤100K',
    '100K–200K',
    '200K–300K',
    '300K–400K',
    '400K–500K',
    '500K–1M',
    '1M–5M',
    '5M–10M',
    '10M–50M',
    '>50M'
]

project_data['budget_band'] = pd.cut(
    project_data[budget_column],
    bins=budget_bins,
    labels=budget_labels,
    include_lowest=True,
    right=True
)

budget_band_summary = (
    project_data['budget_band']
    .value_counts(sort=False)
    .rename('project_count')
    .reset_index()
)

budget_band_summary['project_pct'] = (
    budget_band_summary['project_count']
    .div(len(project_data))
    .mul(100)
)

display(budget_band_summary)

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 6)
)

bars = ax.barh(
    budget_band_summary['budget_band'],
    budget_band_summary['project_pct'],
    color='#4C78A8'
)

ax.bar_label(
    bars,
    labels=[
        f'{value:.1f}%'
        for value
        in budget_band_summary['project_pct']
    ],
    padding=3
)

ax.invert_yaxis()

ax.set_title(
    'Distribution of Construction Projects '
    'by Budget Band'
)

ax.set_xlabel('Share of projects (%)')
ax.set_ylabel('Budget band (THB)')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()
plt.show()

Budget Band Findings

โครงการจำนวน 76.34% มีวงเงินงบประมาณไม่เกิน 500,000 บาท
โดยช่วง 400,000–500,000 บาทเป็นกลุ่มที่มีจำนวนมากที่สุด
คิดเป็น 25.09% ของโครงการทั้งหมด

เมื่อวงเงินสูงกว่า 500,000 บาท สัดส่วนลดลงเหลือเพียง 4.57%
ในช่วง 500,000–1 ล้านบาท ขณะที่โครงการมูลค่าเกิน 50 ล้านบาท
มีเพียง 0.49%

ผลนี้แสดงให้เห็นว่าการพิจารณาเฉพาะจำนวนโครงการอาจทำให้
โครงการขนาดเล็กมีอิทธิพลต่อภาพรวมมากเกินไป ในการวิเคราะห์
ขั้นต่อไปจึงควรพิจารณาทั้งจำนวนโครงการและมูลค่ารวมควบคู่กัน

การกระจุกตัวของวงเงินบริเวณ 500,000 บาทเป็นประเด็นที่น่าสนใจ
สำหรับการวิเคราะห์ร่วมกับวิธีจัดซื้อ แต่ยังไม่สามารถสรุปสาเหตุ
หรือระบุว่าเป็นความผิดปกติได้จากการกระจายของวงเงินเพียงอย่างเดียว

### 3.2 Budget, Reference Price, and Awarded Price

ส่วนนี้เปรียบเทียบราคาที่ตกลงกับวงเงินงบประมาณและราคากลาง
เพื่อดูว่าค่าทั้งสามเคลื่อนไหวสอดคล้องกันเพียงใด และมีโครงการ
ที่อยู่เหนือหรือต่ำกว่าเส้นราคาเท่ากันในลักษณะใด

เนื่องจากข้อมูลมีจำนวนมากและมูลค่ามีช่วงกว้าง จึงสุ่มข้อมูล
10,000 โครงการสำหรับการแสดง scatter plot และใช้ log scale
เพื่อให้มองเห็นทั้งโครงการขนาดเล็กและขนาดใหญ่ได้ในภาพเดียว

การสุ่มใช้ `random_state=42` เพื่อให้ได้ตัวอย่างเดิมเมื่อรันซ้ำ
ส่วนการคำนวณสถิติยังคงใช้ข้อมูลทุกโครงการ

In [ ]:
price_plot_data = (
    project_data
    .loc[
        project_data[
            reference_price_column
        ].notna()
    ]
    .sample(
        n=min(
            10_000,
            project_data[
                reference_price_column
            ].notna().sum()
        ),
        random_state=42
    )
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

comparisons = [
    (
        budget_column,
        'Awarded Price vs Budget'
    ),
    (
        reference_price_column,
        'Awarded Price vs Reference Price'
    )
]

for ax, (x_column, title) in zip(
    axes,
    comparisons
):
    sns.scatterplot(
        data=price_plot_data,
        x=x_column,
        y=awarded_price_column,
        alpha=0.25,
        s=18,
        color='#4C78A8',
        edgecolor=None,
        ax=ax
    )

    lower_limit = min(
        price_plot_data[x_column].min(),
        price_plot_data[
            awarded_price_column
        ].min()
    )

    upper_limit = max(
        price_plot_data[x_column].max(),
        price_plot_data[
            awarded_price_column
        ].max()
    )

    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle='--',
        color='#D62728',
        linewidth=1.5,
        label='Equal price'
    )

    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.set_title(title)
    ax.set_xlabel('Comparison value (THB, log scale)')
    ax.set_ylabel('Awarded price (THB, log scale)')
    ax.legend()

plt.tight_layout()
plt.show()

Price Relationship Findings

ราคาที่ตกลงมีความสัมพันธ์เชิงบวกอย่างชัดเจนกับทั้งวงเงินงบประมาณ
และราคากลาง โดยจุดส่วนใหญ่อยู่ใกล้เส้นราคาเท่ากันหรืออยู่ใต้เส้น
แสดงว่าราคาที่ตกลงมักไม่สูงกว่ากรอบราคาที่ใช้เปรียบเทียบ

การเปรียบเทียบกับราคากลางมีจุดที่อยู่ห่างจากเส้นราคาเท่ากัน
มากกว่าการเปรียบเทียบกับวงเงินงบประมาณ โดยพบทั้งกรณีที่ราคา
ตกลงต่ำกว่าราคากลางมาก และกรณีที่ราคาตกลงสูงกว่าราคากลาง
บางรายการ

กราฟนี้ใช้ข้อมูลตัวอย่าง 10,000 โครงการเพื่อการแสดงผลเท่านั้น
จึงใช้สำหรับมองรูปแบบโดยรวม ไม่ใช้ตัดสินจำนวนหรือสัดส่วน
ของโครงการที่ผิดสังเกต

### 3.3 Comparison by Procurement Method

วิธีจัดซื้ออาจสัมพันธ์กับขนาดโครงการและระดับส่วนต่างราคา
ส่วนนี้จึงเปรียบเทียบจำนวนโครงการ มูลค่า และส่วนต่างราคา
ระหว่างวิธีจัดซื้อทั้ง 4 วิธี

เนื่องจากส่วนต่างร้อยละมีค่าผิดสังเกตรุนแรง การเปรียบเทียบ
ค่ากลางจะใช้ median แทน mean

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
method_column = (
    'ชื่อวิธีการจัดซื้อจัดจ้าง'
)

method_analysis = (
    project_data
    .assign(
        awarded_above_budget=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    budget_column
                ]
            )
        ),
        awarded_equal_budget=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    budget_column
                ]
            )
        ),
        reference_available=(
            project_data[
                reference_price_column
            ]
            .notna()
        ),
        awarded_above_reference=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    reference_price_column
                ]
            )
        ),
        awarded_equal_reference=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    reference_price_column
                ]
            )
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        ),
        above_budget_count=(
            'awarded_above_budget',
            'sum'
        ),
        equal_budget_count=(
            'awarded_equal_budget',
            'sum'
        ),
        reference_available_count=(
            'reference_available',
            'sum'
        ),
        above_reference_count=(
            'awarded_above_reference',
            'sum'
        ),
        equal_reference_count=(
            'awarded_equal_reference',
            'sum'
        )
    )
    .reset_index()
)

method_analysis['project_pct'] = (
    method_analysis['project_count']
    .div(
        method_analysis[
            'project_count'
        ].sum()
    )
    .mul(100)
)

method_analysis['budget_share_pct'] = (
    method_analysis['total_budget']
    .div(
        method_analysis[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

method_analysis['above_budget_pct'] = (
    method_analysis['above_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_budget_pct'] = (
    method_analysis['equal_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['above_reference_pct'] = (
    method_analysis['above_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_reference_pct'] = (
    method_analysis['equal_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis = (
    method_analysis
    .sort_values(
        'project_count',
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
display(
    method_analysis[
        [
            method_column,
            'project_count',
            'project_pct',
            'budget_share_pct',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct',
            'equal_budget_pct',
            'above_budget_pct',
            'equal_reference_pct',
            'above_reference_pct'
        ]
    ]
)

Procurement Method Findings

วิธีเฉพาะเจาะจงเป็นวิธีที่ใช้มากที่สุด คิดเป็น 76.35%
ของจำนวนโครงการทั้งหมด แต่มีสัดส่วนวงเงินรวมเพียง 9.33%
และมีวงเงินงบประมาณมัธยฐาน 283,000 บาท

ในทางตรงกันข้าม e-bidding มีจำนวนเพียง 21.22% ของโครงการ
แต่ครองวงเงินรวมถึง 83.34% และมีวงเงินงบประมาณมัธยฐาน
ประมาณ 3.10 ล้านบาท แสดงว่า e-bidding ถูกใช้กับโครงการ
ที่มีขนาดใหญ่กว่าโดยทั่วไป

ส่วนต่างราคายังแตกต่างกันอย่างชัดเจน โดย e-bidding มีส่วนต่าง
จากงบประมาณมัธยฐาน 8.07% และส่วนต่างจากราคากลางมัธยฐาน
6.60% ขณะที่วิธีเฉพาะเจาะจงมีค่ามัธยฐานเพียง 0.18%
และ 0.22% ตามลำดับ

นอกจากนี้ โครงการเฉพาะเจาะจง 41.76% มีราคาตกลงเท่ากับ
วงเงินงบประมาณ และ 28.17% มีราคาตกลงเท่ากับราคากลาง
จึงควรศึกษาต่อว่ารูปแบบดังกล่าวสัมพันธ์กับช่วงวงเงินหรือไม่

วิธีตกลงราคามีเพียง 2 โครงการ จึงไม่นำมาใช้สรุป
ความแตกต่างระหว่างวิธีจัดซื้อ

In [ ]:
method_label_map = {
    'เฉพาะเจาะจง': 'Specific',
    'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-Bidding',
    'คัดเลือก': 'Selection',
    'ตกลงราคา': 'Price Agreement'
}

method_plot = (
    method_analysis
    .loc[
        method_analysis[
            'project_count'
        ].gt(2)
    ]
    .copy()
)

method_plot['procurement_method'] = (
    method_plot[method_column]
    .map(method_label_map)
)

method_plot = (
    method_plot[
        [
            'procurement_method',
            'project_pct',
            'budget_share_pct'
        ]
    ]
    .melt(
        id_vars='procurement_method',
        var_name='measure',
        value_name='percentage'
    )
)

measure_label_map = {
    'project_pct': 'Share of projects',
    'budget_share_pct': 'Share of total budget'
}

method_plot['measure'] = (
    method_plot['measure']
    .map(measure_label_map)
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

sns.barplot(
    data=method_plot,
    y='procurement_method',
    x='percentage',
    hue='measure',
    palette=[
        '#4C78A8',
        '#E67E22'
    ],
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3
    )

ax.set_title(
    'Project Share and Budget Share '
    'by Procurement Method'
)

ax.set_xlabel('Share (%)')
ax.set_ylabel('Procurement method')
ax.legend(title='')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
budget_concentration = (
    project_data
    .assign(
        budget_le_500k=(
            project_data[
                budget_column
            ].le(500_000)
        ),
        budget_400k_to_500k=(
            project_data[
                budget_column
            ].gt(400_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_490k_to_500k=(
            project_data[
                budget_column
            ].ge(490_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_exactly_500k=(
            project_data[
                budget_column
            ].eq(500_000)
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        budget_le_500k_count=(
            'budget_le_500k',
            'sum'
        ),
        budget_400k_to_500k_count=(
            'budget_400k_to_500k',
            'sum'
        ),
        budget_490k_to_500k_count=(
            'budget_490k_to_500k',
            'sum'
        ),
        budget_exactly_500k_count=(
            'budget_exactly_500k',
            'sum'
        )
    )
    .reset_index()
)

count_columns = [
    'budget_le_500k_count',
    'budget_400k_to_500k_count',
    'budget_490k_to_500k_count',
    'budget_exactly_500k_count'
]

for column in count_columns:
    percentage_column = (
        column
        .replace('_count', '_pct')
    )

    budget_concentration[
        percentage_column
    ] = (
        budget_concentration[column]
        .div(
            budget_concentration[
                'project_count'
            ]
        )
        .mul(100)
    )

display(
    budget_concentration[
        [
            method_column,
            'project_count',
            'budget_le_500k_pct',
            'budget_400k_to_500k_pct',
            'budget_490k_to_500k_pct',
            'budget_exactly_500k_pct'
        ]
    ]
)

Budget Concentration by Procurement Method

การกระจุกตัวของวงเงินบริเวณ 500,000 บาทเกิดขึ้นเด่นชัด
ในโครงการที่ใช้วิธีเฉพาะเจาะจง

โครงการเฉพาะเจาะจงจำนวน 99.57% มีวงเงินไม่เกิน 500,000 บาท
โดย 32.72% อยู่ในช่วงมากกว่า 400,000 ถึง 500,000 บาท
และ 19.14% อยู่ในช่วง 490,000–500,000 บาท นอกจากนี้
4.75% มีวงเงินเท่ากับ 500,000 บาทพอดี

ในทางตรงกันข้าม โครงการที่มีวงเงินไม่เกิน 500,000 บาท
พบเพียง 2.25% ในวิธีคัดเลือก และ 1.19% ใน e-bidding

ผลดังกล่าวแสดงว่าวิธีจัดซื้อมีความสัมพันธ์อย่างมากกับช่วงวงเงิน
และการกระจุกตัวใกล้ 500,000 บาทควรถูกนำไปวิเคราะห์เพิ่มเติม
อย่างไรก็ตาม ข้อมูลนี้ยังไม่เพียงพอที่จะสรุปว่าเกิดการแบ่งโครงการ
หรือการหลีกเลี่ยงกระบวนการจัดซื้อ

### 3.4 Analysis by Government Agency

ส่วนนี้เปรียบเทียบหน่วยงานในสองมิติ ได้แก่ จำนวนโครงการ
และวงเงินงบประมาณรวม เพื่อแยกหน่วยงานที่ดำเนินโครงการจำนวนมาก
ออกจากหน่วยงานที่รับผิดชอบโครงการมูลค่าสูง

การวิเคราะห์ใช้ `ชื่อหน่วยงาน` จากข้อมูลระดับโครงการ
และยังไม่รวม `ชื่อหน่วยงานย่อย`

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
agency_column = 'ชื่อหน่วยงาน'

agency_summary = (
    project_data
    .groupby(agency_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

agency_summary['project_share_pct'] = (
    agency_summary['project_count']
    .div(
        agency_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

agency_summary['budget_share_pct'] = (
    agency_summary['total_budget']
    .div(
        agency_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

agency_summary['total_budget_billion'] = (
    agency_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
top_agencies_by_count = (
    agency_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            agency_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

display(top_agencies_by_count)

In [ ]:
top_agencies_by_budget = (
    agency_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            agency_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

display(top_agencies_by_budget)

**Government Agency Findings**

กรมทางหลวงมีจำนวนโครงการมากที่สุด 5,774 โครงการ และมีวงเงินรวม
สูงที่สุดประมาณ 95.33 พันล้านบาท หรือ 20.03% ของวงเงินทั้งหมด
จึงเป็นหน่วยงานที่มีบทบาทสูงทั้งด้านจำนวนและมูลค่าโครงการ

กรมทางหลวงชนบทและกรมชลประทานอยู่ในอันดับสูงทั้งสองมิติเช่นกัน
ขณะที่กรมโยธาธิการและผังเมืองมีเพียง 780 โครงการ แต่มีวงเงินรวม
สูงถึง 32.31 พันล้านบาท และมีวงเงินมัธยฐาน 20 ล้านบาท
สะท้อนว่าเป็นหน่วยงานที่มีโครงการขนาดใหญ่โดยทั่วไป

ในทางตรงกันข้าม กรมการปกครองและกรมพัฒนาที่ดินมีจำนวนโครงการ
มากเป็นอันดับต้น แต่มีวงเงินมัธยฐานเพียง 373,000 และ 189,600 บาท
ตามลำดับ แสดงให้เห็นว่าการจัดอันดับจากจำนวนโครงการและวงเงินรวม
ให้ภาพที่แตกต่างกัน

หน่วยงาน 4 อันดับแรกตามวงเงินรวมครองวงเงินประมาณ 44.25%
ของข้อมูลทั้งหมด โดยส่วนใหญ่เป็นหน่วยงานที่รับผิดชอบ
โครงสร้างพื้นฐานด้านถนน น้ำ และงานโยธา

In [ ]:
agency_label_map = {
    'กรมทางหลวง': 'Department of Highways',
    'กรมทางหลวงชนบท': 'Department of Rural Roads',
    'กรมชลประทาน': 'Royal Irrigation Department',
    'กรมโยธาธิการและผังเมือง': (
        'Department of Public Works'
    ),
    'การประปาส่วนภูมิภาค': (
        'Provincial Waterworks Authority'
    ),
    'กรุงเทพมหานคร': (
        'Bangkok Metropolitan Administration'
    ),
    'การไฟฟ้านครหลวง': (
        'Metropolitan Electricity Authority'
    ),
    'สำนักงานตำรวจแห่งชาติ': (
        'Royal Thai Police'
    ),
    'กรมทรัพยากรน้ำ': (
        'Department of Water Resources'
    ),
    'กรมทรัพยากรน้ำบาดาล': (
        'Department of Groundwater Resources'
    )
}

agency_budget_plot = (
    agency_summary
    .nlargest(
        10,
        'total_budget'
    )
    .sort_values(
        'total_budget_billion',
        ascending=True
    )
    .copy()
)

agency_budget_plot['agency_label'] = (
    agency_budget_plot[agency_column]
    .map(agency_label_map)
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

bars = ax.barh(
    agency_budget_plot['agency_label'],
    agency_budget_plot[
        'total_budget_billion'
    ],
    color='#4C78A8'
)

ax.bar_label(
    bars,
    labels=[
        f'{budget:.1f}B'
        for budget
        in agency_budget_plot[
            'total_budget_billion'
        ]
    ],
    padding=3
)

ax.set_title(
    'Top 10 Government Agencies '
    'by Construction Budget'
)

ax.set_xlabel('Total budget (billion THB)')
ax.set_ylabel('Government agency')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()
plt.show()

### 3.5 Geographic Distribution

ส่วนนี้เปรียบเทียบจำนวนโครงการและวงเงินรวมในแต่ละจังหวัด
เพื่อดูว่าพื้นที่ที่มีโครงการจำนวนมากเป็นพื้นที่เดียวกับ
พื้นที่ที่ได้รับวงเงินรวมสูงหรือไม่

ข้อมูลจังหวัดระบุสถานที่ของโครงการตามชุดข้อมูลต้นทาง
จึงไม่ได้หมายถึงที่ตั้งสำนักงานใหญ่ของหน่วยงานหรือผู้รับจ้าง

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
province_column = 'จังหวัด'

province_summary = (
    project_data
    .groupby(province_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

province_summary['project_share_pct'] = (
    province_summary['project_count']
    .div(
        province_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

province_summary['budget_share_pct'] = (
    province_summary['total_budget']
    .div(
        province_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

province_summary['total_budget_billion'] = (
    province_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            province_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            province_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

**Geographic Findings**

นครราชสีมามีจำนวนโครงการมากที่สุด 8,641 โครงการ
รองลงมาคืออุบลราชธานี 7,181 โครงการ และขอนแก่น
6,605 โครงการ โดยจังหวัดที่มีโครงการจำนวนมากส่วนใหญ่
มีวงเงินมัธยฐานประมาณ 300,000–400,000 บาท

กรุงเทพมหานครมีจำนวน 4,825 โครงการ หรือ 2.70%
ของโครงการทั้งหมด แต่มีวงเงินรวมสูงถึง 177.54 พันล้านบาท
คิดเป็น 37.30% ของวงเงินทั้งหมด และมีวงเงินมัธยฐาน
4.46 ล้านบาท

ผลนี้แสดงว่าจังหวัดที่มีจำนวนโครงการมากที่สุดไม่จำเป็นต้องเป็น
จังหวัดที่มีวงเงินรวมสูงที่สุด โดยกรุงเทพมหานครมีโครงการ
ขนาดใหญ่กว่าจังหวัดอื่นโดยทั่วไป

การวิเคราะห์พื้นที่จะใช้เป็นบริบทประกอบการอธิบายข้อมูล
แต่จะไม่เป็นแกนหลักของการค้นหา Red Flag ในโครงการนี้

### 3.6 Supplier Data Understanding

การวิเคราะห์ผู้รับจ้างจะใช้ข้อมูลทุกแถวจาก `construction_data`
เนื่องจากโครงการหนึ่งอาจมีหลายสัญญา หลายผู้ชนะ หรือสมาชิก
กิจการร่วมค้า

ก่อนสรุปจำนวนโครงการและมูลค่าตามผู้รับจ้าง จะตรวจสอบว่า
เลขประจำตัวนิติบุคคลสามารถใช้ระบุผู้รับจ้างได้เพียงใด
และมีค่า placeholder หรือความสัมพันธ์แบบหนึ่งรหัสหลายชื่อหรือไม่

In [ ]:
supplier_id_column = (
    'เลขประจำตัวนิติบุคคล 13 หลัก'
)

supplier_name_column = (
    'ชื่อผู้ชนะการเสนอราคา'
)

print('Most frequent supplier IDs:')

display(
    construction_data[
        supplier_id_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

print('\nMost frequent supplier names:')

display(
    construction_data[
        supplier_name_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
supplier_id_profile = (
    construction_data
    .groupby(supplier_id_column)
    [supplier_name_column]
    .nunique()
)

supplier_name_profile = (
    construction_data
    .groupby(supplier_name_column)
    [supplier_id_column]
    .nunique()
)

supplier_relationship_summary = pd.Series({
    'Unique supplier IDs': (
        construction_data[
            supplier_id_column
        ].nunique()
    ),
    'Unique supplier names': (
        construction_data[
            supplier_name_column
        ].nunique()
    ),
    'Supplier IDs linked to multiple names': (
        supplier_id_profile.gt(1).sum()
    ),
    'Supplier names linked to multiple IDs': (
        supplier_name_profile.gt(1).sum()
    )
})

display(
    supplier_relationship_summary
    .to_frame(name='value')
)

**Supplier Identifier Findings**

ข้อมูลมีเลขประจำตัวนิติบุคคลที่ไม่ซ้ำ 33,685 ค่า
และชื่อผู้ชนะการเสนอราคาที่ไม่ซ้ำ 38,826 ค่า

พบเลขนิติบุคคล 4,234 รหัสที่เชื่อมโยงกับชื่อผู้ชนะมากกว่าหนึ่งรูปแบบ
ซึ่งอาจเกิดจากความแตกต่างของการสะกด คำนำหน้า สาขา
หรือรูปแบบการบันทึกชื่อ ขณะเดียวกันพบชื่อผู้ชนะ 253 ชื่อ
ที่เชื่อมโยงกับเลขนิติบุคคลมากกว่าหนึ่งรหัส

ดังนั้น การวิเคราะห์ผู้รับจ้างจะใช้เลขประจำตัวนิติบุคคลเป็นรหัสหลัก
และใช้ชื่อที่พบบ่อยที่สุดของแต่ละรหัสเป็นชื่อสำหรับแสดงผล
โดยจะไม่รวมผู้รับจ้างจากชื่อเพียงอย่างเดียว

In [ ]:
project_awarded_total = (
    project_data[
        awarded_price_column
    ].sum()
)

contract_budget_total = (
    construction_data[
        'วงเงินงบประมาณในสัญญา (บาท)'
    ].sum()
)

value_difference = (
    contract_budget_total
    - project_awarded_total
)

value_difference_pct = (
    value_difference
    / project_awarded_total
    * 100
)

value_reconciliation = pd.Series({
    'Project awarded total': (
        project_awarded_total
    ),
    'Contract budget total': (
        contract_budget_total
    ),
    'Difference': (
        value_difference
    ),
    'Difference pct': (
        value_difference_pct
    ),
    'Missing contract budget rows': (
        construction_data[
            'วงเงินงบประมาณในสัญญา (บาท)'
        ]
        .isna()
        .sum()
    )
})

display(
    value_reconciliation
    .to_frame(name='value')
)

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
contract_value_by_project = (
    construction_data
    .groupby('รหัสโครงการ')
    [
        'วงเงินงบประมาณในสัญญา (บาท)'
    ]
    .sum(
        min_count=1
    )
    .rename('contract_budget_sum')
)

project_value_check = (
    project_data
    [
        [
            'รหัสโครงการ',
            awarded_price_column
        ]
    ]
    .merge(
        contract_value_by_project,
        on='รหัสโครงการ',
        how='left'
    )
)

project_value_check['difference'] = (
    project_value_check[
        'contract_budget_sum'
    ]
    - project_value_check[
        awarded_price_column
    ]
)

project_value_check['is_matched'] = np.isclose(
    project_value_check[
        'contract_budget_sum'
    ],
    project_value_check[
        awarded_price_column
    ],
    rtol=0,
    atol=1
)

project_reconciliation_summary = pd.Series({
    'Projects checked': (
        len(project_value_check)
    ),
    'Matched projects': (
        project_value_check[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~project_value_check[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        project_value_check[
            'is_matched'
        ].mean()
        * 100
    )
})

display(
    project_reconciliation_summary
    .to_frame(name='value')
)

**Contract Value Reconciliation**

ยอดรวมวงเงินระดับสัญญาตรงกับราคาตกลงระดับโครงการจำนวน
178,616 จาก 178,978 โครงการ หรือ 99.80%

มีโครงการที่ยอดไม่ตรงกัน 362 โครงการ และพบวงเงินระดับสัญญา
เป็นค่าว่าง 90 แถว แม้โครงการกลุ่มนี้มีจำนวนเพียง 0.20%
แต่ทำให้ยอดรวมระดับสัญญาสูงกว่าราคาตกลงระดับโครงการ
ประมาณ 13.23 พันล้านบาท หรือ 2.94%

ดังนั้น การวิเคราะห์มูลค่าตามผู้รับจ้างจะใช้เฉพาะโครงการ
ที่ผ่านการ Reconciliation เพื่อป้องกันการรวมมูลค่าซ้ำหรือ
ใช้ข้อมูลสัญญาที่ไม่สอดคล้องกัน

ส่วนโครงการที่ Reconciliation ไม่ผ่านจะไม่ถูกลบจากข้อมูลต้นทาง
และจะถูกเก็บไว้เป็น Candidate Data-quality Red Flag
สำหรับการตรวจสอบใน Notebook 03

In [ ]:
matched_project_ids = set(
    project_value_check.loc[
        project_value_check[
            'is_matched'
        ],
        'รหัสโครงการ'
    ]
)

supplier_value_data = (
    construction_data
    .loc[
        construction_data[
            'รหัสโครงการ'
        ].isin(
            matched_project_ids
        )
        & construction_data[
            'วงเงินงบประมาณในสัญญา (บาท)'
        ].notna()
    ]
    .copy()
)

print(
    f'Supplier value rows: '
    f'{len(supplier_value_data):,}'
)

print(
    f'Reconciled projects: '
    f'{supplier_value_data["รหัสโครงการ"].nunique():,}'
)

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
supplier_name_lookup = (
    construction_data
    .groupby(
        [
            supplier_id_column,
            supplier_name_column
        ]
    )
    .size()
    .rename('name_count')
    .reset_index()
    .sort_values(
        [
            supplier_id_column,
            'name_count',
            supplier_name_column
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .drop_duplicates(
        subset=supplier_id_column,
        keep='first'
    )
    [
        [
            supplier_id_column,
            supplier_name_column
        ]
    ]
    .rename(
        columns={
            supplier_name_column:
            'supplier_display_name'
        }
    )
)

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
supplier_project_summary = (
    construction_data
    .groupby(supplier_id_column)
    .agg(
        supplier_project_count=(
            'รหัสโครงการ',
            'nunique'
        ),
        supplier_record_count=(
            'รหัสโครงการ',
            'size'
        )
    )
    .reset_index()
)

supplier_value_summary = (
    supplier_value_data
    .groupby(supplier_id_column)
    .agg(
        total_contract_value=(
            'วงเงินงบประมาณในสัญญา (บาท)',
            'sum'
        ),
        median_contract_value=(
            'วงเงินงบประมาณในสัญญา (บาท)',
            'median'
        )
    )
    .reset_index()
)

supplier_summary = (
    supplier_project_summary
    .merge(
        supplier_value_summary,
        on=supplier_id_column,
        how='left'
    )
    .merge(
        supplier_name_lookup,
        on=supplier_id_column,
        how='left'
    )
)

supplier_summary[
    'total_contract_value'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .fillna(0)
)

supplier_summary[
    'total_contract_value_million'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(1_000_000)
)

supplier_summary[
    'value_share_pct'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(
        supplier_summary[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

In [ ]:
top_suppliers_by_projects = (
    supplier_summary
    .nlargest(
        15,
        'supplier_project_count'
    )
    [
        [
            supplier_id_column,
            'supplier_display_name',
            'supplier_project_count',
            'supplier_record_count',
            'total_contract_value_million'
        ]
    ]
)

display(top_suppliers_by_projects)

In [ ]:
top_suppliers_by_value = (
    supplier_summary
    .nlargest(
        15,
        'total_contract_value'
    )
    [
        [
            supplier_id_column,
            'supplier_display_name',
            'total_contract_value_million',
            'value_share_pct',
            'supplier_project_count',
            'median_contract_value'
        ]
    ]
)

display(top_suppliers_by_value)

**Supplier Findings**

ผู้รับจ้างที่ได้รับโครงการจำนวนมากที่สุดไม่ใช่ผู้รับจ้าง
ที่ได้รับมูลค่ารวมสูงที่สุด

ห้างหุ้นส่วนจำกัด ตั้นอ้าย วิศวกรรม ได้รับโครงการมากที่สุด
575 โครงการ แต่มีมูลค่ารวมประมาณ 284.68 ล้านบาท
ขณะที่บริษัท วงษ์สยามก่อสร้าง จำกัด มีเพียง 3 โครงการ
แต่มีมูลค่ารวมสูงที่สุดประมาณ 6.82 พันล้านบาท

รายชื่อผู้รับจ้างอันดับต้นตามจำนวนโครงการและตามมูลค่ารวม
แตกต่างกันเกือบทั้งหมด แสดงให้เห็นรูปแบบผู้รับจ้างสองกลุ่ม:

1. ผู้รับจ้างที่ได้รับโครงการขนาดเล็กจำนวนมาก
2. ผู้รับจ้างที่ได้รับโครงการขนาดใหญ่มูลค่าสูงเพียงไม่กี่รายการ

ผู้รับจ้างที่มีมูลค่าสูงที่สุดมีสัดส่วน 1.56% ของมูลค่าสัญญา
ที่ผ่านการ Reconciliation จึงยังไม่สามารถสรุปการกระจุกตัว
จากผู้รับจ้างอันดับหนึ่งเพียงรายเดียวได้ จำเป็นต้องวิเคราะห์
มูลค่าสะสมของผู้รับจ้างทั้งหมด

In [ ]:
supplier_pareto = (
    supplier_summary
    .loc[
        supplier_summary[
            'total_contract_value'
        ].gt(0)
    ]
    .sort_values(
        'total_contract_value',
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

supplier_pareto['supplier_rank'] = (
    supplier_pareto.index + 1
)

supplier_pareto['cumulative_value'] = (
    supplier_pareto[
        'total_contract_value'
    ].cumsum()
)

supplier_pareto[
    'cumulative_value_pct'
] = (
    supplier_pareto[
        'cumulative_value'
    ]
    .div(
        supplier_pareto[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

total_suppliers = len(
    supplier_pareto
)

supplier_count_50 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(50)
    .idxmax()
    + 1
)

supplier_count_80 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(80)
    .idxmax()
    + 1
)

supplier_count_90 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(90)
    .idxmax()
    + 1
)

In [ ]:
supplier_concentration_summary = pd.Series({
    'Suppliers with reconciled value': (
        total_suppliers
    ),
    'Top 1 supplier value share (%)': (
        supplier_pareto
        .head(1)[
            'value_share_pct'
        ]
        .sum()
    ),
    'Top 10 suppliers value share (%)': (
        supplier_pareto
        .head(10)[
            'value_share_pct'
        ]
        .sum()
    ),
    'Top 100 suppliers value share (%)': (
        supplier_pareto
        .head(100)[
            'value_share_pct'
        ]
        .sum()
    ),
    'Suppliers accounting for 50% of value': (
        supplier_count_50
    ),
    'Supplier pct accounting for 50%': (
        supplier_count_50
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 80% of value': (
        supplier_count_80
    ),
    'Supplier pct accounting for 80%': (
        supplier_count_80
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 90% of value': (
        supplier_count_90
    ),
    'Supplier pct accounting for 90%': (
        supplier_count_90
        / total_suppliers
        * 100
    )
})

display(
    supplier_concentration_summary
    .to_frame(name='value')
)

**Supplier Concentration Findings**

ผู้รับจ้างที่มีข้อมูลมูลค่าสัญญาซึ่งผ่านการ Reconciliation
มีทั้งหมด 33,238 ราย

ผู้รับจ้างรายใหญ่ที่สุดครองมูลค่า 1.56% ขณะที่ผู้รับจ้าง
10 อันดับแรกครองมูลค่ารวม 7.67% และ 100 อันดับแรก
ครองมูลค่า 28.24%

เมื่อพิจารณามูลค่าสะสม พบว่า:

- ผู้รับจ้าง 391 ราย หรือ 1.18% ครองมูลค่า 50%
- ผู้รับจ้าง 1,796 ราย หรือ 5.40% ครองมูลค่า 80%
- ผู้รับจ้าง 3,767 ราย หรือ 11.33% ครองมูลค่า 90%

ผลนี้แสดงว่ามูลค่าสัญญากระจุกตัวอยู่ในผู้รับจ้างส่วนน้อย
แต่ไม่ได้ถูกครองโดยผู้รับจ้างรายเดียวหรือกลุ่ม Top 10
อย่างรุนแรง การประเมินความเสี่ยงจึงไม่ควรพิจารณาเพียง
อันดับมูลค่ารวม แต่ควรพิจารณาจำนวนโครงการ ขนาดโครงการ
หน่วยงาน และวิธีจัดซื้อร่วมกัน

In [ ]:
supplier_pareto[
    'cumulative_supplier_pct'
] = (
    supplier_pareto[
        'supplier_rank'
    ]
    .div(total_suppliers)
    .mul(100)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    supplier_pareto[
        'cumulative_supplier_pct'
    ],
    supplier_pareto[
        'cumulative_value_pct'
    ],
    color='#4C78A8',
    linewidth=2
)

ax.axhline(
    y=80,
    color='#D62728',
    linestyle='--',
    linewidth=1.5,
    label='80% of contract value'
)

ax.axvline(
    x=(
        supplier_count_80
        / total_suppliers
        * 100
    ),
    color='#E67E22',
    linestyle='--',
    linewidth=1.5,
    label=(
        f'{supplier_count_80:,} suppliers '
        f'({supplier_count_80 / total_suppliers * 100:.1f}%)'
    )
)

ax.scatter(
    supplier_count_80
    / total_suppliers
    * 100,
    80,
    color='#D62728',
    s=60,
    zorder=3
)

ax.set_title(
    'Cumulative Contract Value '
    'by Supplier Share'
)

ax.set_xlabel(
    'Cumulative share of suppliers (%)'
)

ax.set_ylabel(
    'Cumulative share of contract value (%)'
)

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.legend(
    loc='lower right'
)

ax.spines[
    ['top', 'right']
].set_visible(False)

plt.tight_layout()
plt.show()

## 4. EDA Summary and Candidate Findings

การวิเคราะห์เชิงสำรวจพบรูปแบบสำคัญสำหรับนำไปศึกษาเพิ่มเติม
ใน Notebook การวิเคราะห์ Red Flag ดังนี้

### 1. Project-value Distribution

โครงการจ้างก่อสร้างมีวงเงินงบประมาณแบบเบ้ขวาอย่างมาก
โดย 76.34% ของโครงการมีวงเงินไม่เกิน 500,000 บาท
ขณะที่โครงการมูลค่าเกิน 50 ล้านบาทมีเพียง 0.49%

แม้โครงการขนาดใหญ่มีจำนวนน้อย แต่มีอิทธิพลต่อมูลค่ารวมสูง
จึงไม่ควรประเมินความสำคัญจากจำนวนโครงการเพียงอย่างเดียว

### 2. Concentration around 500,000 Baht

โครงการเฉพาะเจาะจงจำนวน 99.57% มีวงเงินไม่เกิน 500,000 บาท
และ 19.14% อยู่ในช่วง 490,000–500,000 บาท

การกระจุกตัวบริเวณดังกล่าวเป็น Candidate Finding ที่ควรตรวจสอบ
ร่วมกับหน่วยงาน ผู้รับจ้าง ชื่อโครงการ และช่วงเวลาของรายการ
แต่ยังไม่สามารถสรุปว่าเกิดการแบ่งโครงการหรือหลีกเลี่ยงขั้นตอนจัดซื้อ

### 3. Procurement-method Differences

วิธีเฉพาะเจาะจงคิดเป็น 76.35% ของจำนวนโครงการ
แต่มีวงเงินเพียง 9.33% ของทั้งหมด

e-bidding มีจำนวนเพียง 21.22% ของโครงการ แต่ครองวงเงิน
ถึง 83.34% และมีส่วนต่างจากราคากลางมัธยฐาน 6.60%
สูงกว่าวิธีเฉพาะเจาะจงซึ่งมีค่ามัธยฐาน 0.22%

### 4. Agency and Geographic Differences

กรมทางหลวงมีวงเงินรวมสูงที่สุดประมาณ 95.33 พันล้านบาท
หรือ 20.03% ของวงเงินทั้งหมด

กรุงเทพมหานครมีเพียง 2.70% ของจำนวนโครงการ
แต่ครองวงเงินรวม 37.30% แสดงว่าจำนวนโครงการและมูลค่า
ให้ภาพการกระจายที่แตกต่างกัน

ข้อมูลพื้นที่จะใช้เป็นบริบทประกอบ แต่ไม่ใช่แกนหลักของ Red Flag

### 5. Supplier Concentration

ผู้รับจ้างที่ได้โครงการจำนวนมากที่สุดไม่ใช่ผู้รับจ้าง
ที่ได้รับมูลค่ารวมสูงที่สุด

ผู้รับจ้างเพียง 1,796 ราย หรือ 5.40% ครองมูลค่าสัญญา
80% ของข้อมูลที่ผ่านการ Reconciliation อย่างไรก็ตาม
ผู้รับจ้าง 10 อันดับแรกครองมูลค่าเพียง 7.67%
จึงไม่ใช่การกระจุกตัวในผู้รับจ้างไม่กี่รายเท่านั้น

### 6. Data-quality Findings

มี 362 โครงการ หรือ 0.20% ที่ผลรวมวงเงินระดับสัญญา
ไม่ตรงกับราคาตกลงระดับโครงการ และพบส่วนต่างราคาร้อยละ
ที่สูงผิดสังเกตจากราคากลางซึ่งมีค่าต่ำมากบางรายการ

กรณีเหล่านี้ควรถูกแยกเป็น Data-quality Red Flag
ไม่ควรรวมกับ Procurement Red Flag โดยอัตโนมัติ

## Candidate Analyses for the Next Notebook

1. โครงการเฉพาะเจาะจงที่มีวงเงินใกล้ 500,000 บาท
2. โครงการที่ราคาตกลงสูงกว่างบประมาณ
3. โครงการที่ราคาตกลงสูงกว่าราคากลาง
4. โครงการที่มีส่วนต่างราคาร้อยละรุนแรงจาก denominator ผิดสังเกต
5. โครงการที่ยอดรวมระดับสัญญาไม่ตรงกับราคาตกลงระดับโครงการ
6. ผู้รับจ้างที่ได้หลายโครงการจากหน่วยงานหรือพื้นที่เดียวกัน
7. โครงการที่อาจมีชื่อหรือรายละเอียดคล้ายกันและเกิดในช่วงใกล้กัน

Candidate เหล่านี้เป็นหัวข้อสำหรับตรวจสอบต่อ
ยังไม่ใช่เกณฑ์ยืนยันการทุจริต

In [ ]:
project_output_path = (
    data_path.parent
    / 'construction_projects_2569.csv'
)

pass  # GitHub: แสดงผลใน Notebook ไม่ส่งออกไฟล์

print("แสดงผลใน Notebook; ไม่บันทึกไฟล์ผลลัพธ์")


In [ ]:
supplier_output_path = (
    data_path.parent
    / 'construction_supplier_summary_2569.csv'
)

pass  # GitHub: แสดงผลใน Notebook ไม่ส่งออกไฟล์

print("แสดงผลใน Notebook; ไม่บันทึกไฟล์ผลลัพธ์")


In [ ]:
# ผลลัพธ์ยังอยู่ใน DataFrame สำหรับดูต่อ ไม่ต้องเขียน CSV
display(pd.DataFrame({"ตาราง": ["project_data", "supplier_summary"], "จำนวนแถว": [len(project_data), len(supplier_summary)]}))


## 5. Output for Red Flag Analysis

Notebook นี้สร้างข้อมูลสำหรับใช้ในขั้นต่อไปจำนวน 2 ไฟล์:

1. `construction_projects_2569.csv`  
   ข้อมูลระดับโครงการ พร้อมตัวแปรส่วนต่างราคาและช่วงวงเงิน

2. `construction_supplier_summary_2569.csv`  
   ข้อมูลสรุประดับผู้รับจ้าง โดยใช้เลขประจำตัวนิติบุคคลเป็นรหัสหลัก

Notebook ถัดไปจะใช้ผลจาก EDA นี้กำหนด Red Flag
โดยแยก Procurement Red Flag ออกจาก Data-quality Red Flag
และตรวจสอบกรณีศึกษาจากข้อมูลจริง

In [ ]:
# คืนพื้นที่ชั่วคราวและปิดรูป โดยเก็บ DataFrame ผลวิเคราะห์ไว้
cleanup_runtime()
